# Multi-embedding → Source attention → BiLSTM → CRF
Một chuỗi token theo regex (từ + dấu câu), có offset ký tự, dùng giống nhau khi train và predict.
BiomedBERT chạy encoder contextual, mean-pool WordPiece về token. Flair PubMed forward/backward
là một nguồn; fastText là một nguồn. Mỗi nguồn có projection riêng về 200 chiều; softmax theo nguồn
ở mỗi token rồi cộng có trọng số. Encoder đóng băng; projection, attention, BiLSTM và CRF được học.

Chạy tuần tự; sửa đường dẫn dữ liệu và ENABLED_SOURCES. Tắt fasttext để thử nhanh hai nguồn.
Audit annotation trước khi tải checkpoint. ANNOTATION_POLICY='project' mở rộng span tới
ranh giới token, gộp annotation trùng hệt, ưu tiên span dài hơn khi tranh cùng token
(hòa: start/end/type); span thua bị bỏ toàn bộ. Giữ câu/split/tokenizer nguyên vẹn,
không chia token dựa trên nhãn gold. Mọi điều chỉnh được ghi annotation_report.json
kèm thống kê từng loại. F1 đo nhãn BIO đã project, không phải toàn bộ annotation gốc.
Dùng ANNOTATION_POLICY='strict' nếu cần dừng trước mọi điều chỉnh; offset sai/câu rỗng
vẫn luôn báo lỗi. Chưa thể giữ đầy đủ thực thể chồng lấn với CRF một nhãn/token.

Transformer chia câu dài thành cửa sổ tại ranh giới token, không cắt mất token;
context Transformer bị giới hạn trong từng cửa sổ, BiLSTM vẫn nhận cả câu.
F1 là strict BIO theo token, các span rời rạc được đánh giá riêng; không phải official discontinuous score.
CRF không áp ràng buộc BIO cứng. Nhãn I sai được coi như B chỉ khi hiển thị dự đoán.

Nguồn: [Hugging Face word alignment](https://huggingface.co/docs/transformers/main_classes/tokenizer),
[Flair](https://flairnlp.github.io/docs/tutorial-embeddings/flair-embeddings),
[fastText](https://huggingface.co/facebook/fasttext-en-vectors).


In [2]:
!pip install "google-cloud-bigquery-storage>=2.0.0"

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.0/308.0 kB 4.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 67.6 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.81.1
    Uninstalling grpcio-1.81.1:
      Successfully uninstalled grpcio-1.81.1
  Attempting uninstall: grpcio-status
    Found existing installation: grpcio-status 1.71.2
    Uninstalling grpcio-status-1.71.2:
      Successfully uninstalled grpcio-status-1.71.2
ERROR: pip's dependency resolver does not currently take into account all the packages that 

In [3]:
!pip install flair


  Using cached flair-0.15.1-py3-none-any.whl.metadata (12 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.2 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.6 MB/s eta 0:00:00
  Created wheel for langdetect: filename=lang

In [4]:
!pip install --upgrade pip setuptools wheel
!pip install seqeval evaluate tqdm pytorch-crf fasttext-wheel huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 38.1 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
  Using cached seqeval-1.2.2.tar.gz (43 kB)
  Installing build dependencies ... done
 

In [13]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import random, tempfile
import xml.etree.ElementTree as ET
from pathlib import Path
import torch
SEED = 23022006
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ENABLED_SOURCES = ['biomedbert', 'flair', 'fasttext']
# Thử nhanh: ENABLED_SOURCES = ['biomedbert', 'flair']
MODEL_NAME = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
FLAIR_NAMES = ['pubmed-forward', 'pubmed-backward']
TRANSFORMER_MAX_LENGTH = 256
EPOCHS, PATIENCE, BATCH_SIZE, LR = 40, 7, 32, 1e-3
ANNOTATION_POLICY = 'project'  # 'strict': dừng khi annotation không khớp token BIO
RUN_NAME = '-'.join(ENABLED_SOURCES) + '-' + ANNOTATION_POLICY
OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')) / 'multi_embedding_fusion' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Có thể trỏ tới model.bin đã gắn từ Kaggle Dataset.
FASTTEXT_PATH = OUTPUT_DIR.parent / 'pretrained' / 'fasttext-en-vectors' / 'model.bin'
DOWNLOAD_FASTTEXT = True
ENTITY_TYPES = ['drug', 'brand', 'group', 'drug_n']
LABELS = ['O'] + [f'{p}-{t}' for t in ENTITY_TYPES for p in ('B', 'I')]
label2id = {label: i for i, label in enumerate(LABELS)}
if not ENABLED_SOURCES or len(set(ENABLED_SOURCES)) != len(ENABLED_SOURCES) or set(ENABLED_SOURCES) - {'biomedbert', 'flair', 'fasttext'}:
    raise ValueError('ENABLED_SOURCES không hợp lệ')
print(DEVICE, ENABLED_SOURCES)


cuda ['biomedbert', 'flair', 'fasttext']


In [14]:
import json
import re
import hashlib
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torchcrf import CRF

TOKEN_PATTERN = r"\w+|[^\w\s]"


def tokenize_words(text):
    matches = list(re.finditer(TOKEN_PATTERN, text))
    return [m.group() for m in matches], [(m.start(), m.end()) for m in matches]


def prepare_example(ex, label2id, policy="strict"):
    if policy not in {"strict", "project"}:
        raise ValueError(f"Unknown annotation policy: {policy}")
    words, offsets = tokenize_words(ex['text'])
    labels = [label2id['O']] * len(words)
    owners, issues = {}, []
    if not words:
        issues.append(dict(reason='empty_sentence', fatal=True))
    seen = set()
    # Deterministic longest-span-first projection, independent of XML entity order.
    entities = sorted(ex['entities'], key=lambda e: (-(e['end']-e['start']), e['start'], e['end'], e['type']))
    for ent in entities:
        if not 0 <= ent['start'] < ent['end'] <= len(ex['text']):
            issues.append(dict(reason='invalid_offset', entity=ent, fatal=True))
            continue
        key = (ent['start'], ent['end'], ent['type'])
        if key in seen:
            issues.append(dict(reason='duplicate_annotation', entity=ent, action='deduplicate'))
            continue
        seen.add(key)
        positions = [i for i, (s, e) in enumerate(offsets) if s < ent['end'] and e > ent['start']]
        if not positions:
            issues.append(dict(reason='no_token_for_entity', entity=ent, fatal=True))
            continue
        projected = dict(start=offsets[positions[0]][0], end=offsets[positions[-1]][1], type=ent['type'])
        if projected['start'] != ent['start'] or projected['end'] != ent['end']:
            issues.append(dict(reason='boundary_inside_token', entity=ent, projected=projected,
                               token_offsets=[offsets[i] for i in positions], action='expand_to_token_boundaries'))
            if policy == 'strict':
                continue
        shared = [i for i in positions if i in owners]
        if shared:
            issues.append(dict(reason='overlapping_entities', entity=ent,
                               conflicting_entities=[owners[i] for i in shared], action='drop_conflicting_span'))
            continue
        for j, i in enumerate(positions):
            labels[i] = label2id[f"{'B' if j == 0 else 'I'}-{ent['type']}"]
            owners[i] = ent
    return dict(text=ex['text'], words=words, offsets=offsets, labels=labels), issues


def audit_splits(splits, label2id, report_path, policy="strict"):
    from collections import Counter
    import warnings
    if policy not in {"strict", "project"}:
        raise ValueError(f"Unknown annotation policy: {policy}")
    prepared, report, summary = {}, {}, {}
    for name, examples in splits.items():
        prepared[name], report[name] = [], []
        for ex in examples:
            row, issues = prepare_example(ex, label2id, policy=policy)
            prepared[name].append(row)
            if issues:
                report[name].append(dict(id=ex.get('id'), document=ex.get('document'),
                                         text=ex['text'], issues=issues))
        counts = Counter(issue['reason'] for row in report[name] for issue in row['issues'])
        summary[name] = dict(sentences=len(examples), affected_sentences=len(report[name]),
                             reasons=dict(counts))
        print(f"{name}: {len(report[name])}/{len(examples)} câu cần xử lý; {dict(counts)}")
    report['_policy'] = policy
    report['_summary'] = summary
    Path(report_path).write_text(json.dumps(report, ensure_ascii=False, indent=2))
    affected = sum(info['affected_sentences'] for info in summary.values())
    fatal = any(issue.get('fatal', False) for name in splits for row in report[name] for issue in row['issues'])
    if fatal or (policy == 'strict' and affected):
        raise ValueError(f'Annotation không biểu diễn chính xác bằng token BIO. Xem {report_path}. '
                         'Chế độ project chỉ xử lý ranh giới/trùng/xung đột; không bỏ qua offset sai hoặc câu rỗng.')
    if affected:
        warnings.warn(f'Đã project annotation trong {affected} câu; xem {report_path}. '
                      'F1 dùng nhãn đã điều chỉnh, không phải toàn bộ annotation gốc.')
    return prepared


def word_chunks(words, tokenizer, max_length):
    budget = max_length - tokenizer.num_special_tokens_to_add(pair=False)
    start, used = 0, 0
    for i, word in enumerate(words):
        size = len(tokenizer([word], is_split_into_words=True, add_special_tokens=False)['input_ids'])
        if size < 1 or size > budget:
            raise ValueError(f'Token không vừa cửa sổ Transformer: {word!r}, {size} pieces')
        if used + size > budget:
            yield start, i
            start, used = i, 0
        used += size
    if start < len(words):
        yield start, len(words)




In [15]:
TRAIN_DIR_DrugBank = Path("/kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Train/DrugBank")
TRAIN_DIR_MedLine = Path("/kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Train/MedLine")

TEST_ROOT = Path("/kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Test")
TEST_DIR = TEST_ROOT / "Test for DrugNER task"


In [16]:
from pathlib import Path
from sklearn.model_selection import train_test_split


# 1. Chia danh sách file của từng nguồn
def split_xml_files(folder, dev_size=0.1, seed=42):
    xml_files = sorted(Path(folder).rglob("*.xml"))

    train_files, dev_files = train_test_split(
        xml_files,
        test_size=dev_size,
        random_state=seed,
        shuffle=True,
    )

    return train_files, dev_files
train_files_drug, dev_files_drug = split_xml_files(
    TRAIN_DIR_DrugBank, seed= SEED
)

train_files_med, dev_files_med = split_xml_files(
    TRAIN_DIR_MedLine, seed= SEED
)
test_files = sorted(TEST_DIR.rglob("*.xml"))

In [17]:
def read_xml_file(path):
    examples = []
    for sent in ET.parse(path).getroot().iter("sentence"):
        text = sent.attrib.get("text", "")
        if not text.strip():
            if sent.findall("entity"):
                raise ValueError(f"Annotation trong câu rỗng: {path}")
            continue
        entities = []
        for ent in sent.findall("entity"):
            kind = ent.attrib["type"]
            if kind not in ENTITY_TYPES:
                raise ValueError(f"Loại entity không hỗ trợ: {kind}")
            for span in ent.attrib["charOffset"].split(";"):
                start, last = map(int, span.split("-"))
                end = last + 1
                if not 0 <= start < end <= len(text):
                    raise ValueError(f"Offset XML không hợp lệ: {path}, {span}")
                entities.append(dict(start=start, end=end, type=kind, text=text[start:end]))
        examples.append(dict(id=sent.attrib.get("id", ""), document=str(path),
                             text=text, entities=entities))
    return examples

# 2. Đọc các câu từ danh sách file đã chia
def load_examples_from_files(xml_files):
    examples = []

    for xml_file in xml_files:
        examples.extend(read_xml_file(xml_file))

    return examples
train_examples_drug = load_examples_from_files(train_files_drug)
dev_examples_drug = load_examples_from_files(dev_files_drug)

train_examples_med = load_examples_from_files(train_files_med)
dev_examples_med = load_examples_from_files(dev_files_med)
# 3. Gộp hai nguồn
train_examples = train_examples_drug + train_examples_med
dev_examples = dev_examples_drug + dev_examples_med

# 4. Giữ nguyên tập test có sẵn
test_examples = load_examples_from_files(test_files)

In [18]:
# 5. Kiểm tra không trùng file giữa train và dev
train_files = train_files_drug + train_files_med
dev_files = dev_files_drug + dev_files_med

assert set(train_files).isdisjoint(dev_files)

for source, train_part, dev_part in [
    ("DrugBank", train_examples_drug, dev_examples_drug),
    ("MedLine", train_examples_med, dev_examples_med),
]:
    total = len(train_part) + len(dev_part)
    print(
        f"{source}: train={len(train_part)} câu | "
        f"dev={len(dev_part)} câu | "
        f"dev chiếm {len(dev_part) / total:.2%} số câu"
    )

print(f"\nTổng train: {len(train_examples)} câu")
print(f"Tổng dev:   {len(dev_examples)} câu")
print(f"Tổng test:  {len(test_examples)} câu")
manifest = dict(seed=SEED, unit="xml_document", dev_fraction=0.1,
                train=[str(p) for p in train_files], dev=[str(p) for p in dev_files],
                test=[str(p) for p in test_files])
(OUTPUT_DIR / "split_manifest.json").write_text(json.dumps(manifest, indent=2))

DrugBank: train=4928 câu | dev=676 câu | dev chiếm 12.06% số câu
MedLine: train=1124 câu | dev=177 câu | dev chiếm 13.60% số câu

Tổng train: 6052 câu
Tổng dev:   853 câu
Tổng test:  665 câu


78508

In [19]:
if not train_examples or not dev_examples:
    raise ValueError('Train/dev rỗng; kiểm tra đường dẫn XML')
prepared = audit_splits(dict(train=train_examples, dev=dev_examples, test=test_examples),
                        label2id, OUTPUT_DIR / 'annotation_report.json', policy=ANNOTATION_POLICY)


train: 66/6052 câu cần xử lý; {'boundary_inside_token': 50, 'overlapping_entities': 24, 'duplicate_annotation': 9}
dev: 11/853 câu cần xử lý; {'boundary_inside_token': 9, 'overlapping_entities': 5}
test: 2/665 câu cần xử lý; {'boundary_inside_token': 2, 'overlapping_entities': 2}


/tmp/ipykernel_58/1679285846.py:87: UserWarning: Đã project annotation trong 79 câu; xem /kaggle/working/multi_embedding_fusion/biomedbert-flair-fasttext-project/annotation_report.json. F1 dùng nhãn đã điều chỉnh, không phải toàn bộ annotation gốc.
  warnings.warn(f'Đã project annotation trong {affected} câu; xem {report_path}. '


In [21]:
class TransformerSource:
    def __init__(self, model_name, device, max_length=256):
        from transformers import AutoModel, AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        if not self.tokenizer.is_fast:
            raise ValueError('Cần fast tokenizer có word_ids')
        self.model = AutoModel.from_pretrained(model_name).eval().requires_grad_(False).to(device)
        self.device = device
        self.dim = self.model.config.hidden_size
        self.max_length = min(max_length, self.model.config.max_position_embeddings)

    @torch.no_grad()
    def encode(self, words):
        self.model.eval()
        rows = []
        for start, end in word_chunks(words, self.tokenizer, self.max_length):
            encoded = self.tokenizer(words[start:end], is_split_into_words=True,
                                     return_tensors='pt', truncation=False)
            if encoded['input_ids'].shape[1] > self.max_length:
                raise ValueError('Chunk vượt giới hạn Transformer')
            word_ids = encoded.word_ids()
            hidden = self.model(**{k: v.to(self.device) for k, v in encoded.items()}).last_hidden_state[0]
            for i in range(end - start):
                indices = [j for j, word_id in enumerate(word_ids) if word_id == i]
                if not indices:
                    raise ValueError('Tokenizer làm mất token')
                rows.append(hidden[indices].mean(0).cpu())
        return torch.stack(rows).float()


class FastTextSource:
    def __init__(self, model_path):
        import fasttext
        self.model = fasttext.load_model(str(model_path))
        self.dim = self.model.get_dimension()

    def encode(self, words):
        import numpy as np
        return torch.from_numpy(np.stack([self.model.get_word_vector(w.lower()) for w in words])).float()


class FlairSource:
    def __init__(self, names, device):
        import flair
        from flair.embeddings import FlairEmbeddings, StackedEmbeddings
        flair.device = device
        self.model = StackedEmbeddings([FlairEmbeddings(n, fine_tune=False) for n in names])
        self.model.eval().requires_grad_(False).to(device)
        self.dim = self.model.embedding_length

    @torch.no_grad()
    def encode(self, words):
        from flair.data import Sentence
        sentence = Sentence(words)
        self.model.eval()
        self.model.embed(sentence)
        result = torch.stack([t.embedding.detach().cpu() for t in sentence]).float()
        sentence.clear_embeddings()
        return result


class CachedDataset(Dataset):
    def __init__(self, rows, sources, cache_dir):
        from tqdm.auto import tqdm
        self.rows, self.paths = rows, []
        self.names = list(sources)
        Path(cache_dir).mkdir(parents=True, exist_ok=True)
        for row in tqdm(rows, desc='Cache embeddings theo token'):
            digest = hashlib.sha256(row['text'].encode()).hexdigest()
            path = Path(cache_dir) / f'{digest}.pt'
            if not path.exists():
                features = {name: source.encode(row['words']) for name, source in sources.items()}
                for name, value in features.items():
                    if value.shape != (len(row['words']), sources[name].dim) or not torch.isfinite(value).all():
                        raise ValueError(f'Feature không hợp lệ: {name}')
                torch.save(dict(words=row['words'], features=features), path)
            self.paths.append(path)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        cached = torch.load(self.paths[index], map_location='cpu', weights_only=True)
        if cached['words'] != row['words'] or list(cached['features']) != self.names:
            raise ValueError('Cache không khớp token hoặc nguồn')
        return dict(features=cached['features'], tags=torch.tensor(row['labels'], dtype=torch.long))


def collate_words(items):
    names = list(items[0]['features'])
    lengths = torch.tensor([len(item['tags']) for item in items])
    if (lengths <= 0).any():
        raise ValueError('CRF không nhận câu rỗng')
    features = {name: pad_sequence([item['features'][name] for item in items], batch_first=True)
                for name in names}
    tags = pad_sequence([item['tags'] for item in items], batch_first=True)
    mask = torch.arange(tags.size(1))[None, :] < lengths[:, None]
    return dict(features=features, tags=tags, mask=mask, lengths=lengths)


def move_batch(batch, device):
    return dict(features={k: v.to(device) for k, v in batch['features'].items()},
                tags=batch['tags'].to(device), mask=batch['mask'].to(device), lengths=batch['lengths'])


class SourceAttention(nn.Module):
    def __init__(self, source_dims, fusion_dim):
        super().__init__()
        if not source_dims:
            raise ValueError('Cần ít nhất một nguồn embedding')
        self.names = list(source_dims)
        self.projections = nn.ModuleDict({name: nn.Sequential(nn.Linear(dim, fusion_dim),
                                         nn.LayerNorm(fusion_dim)) for name, dim in source_dims.items()})
        self.source_bias = nn.Parameter(torch.zeros(len(self.names), fusion_dim))
        self.score = nn.Sequential(nn.Linear(fusion_dim, fusion_dim), nn.Tanh(), nn.Linear(fusion_dim, 1, bias=False))

    def forward(self, features, mask):
        projected = torch.stack([self.projections[name](features[name]) for name in self.names], dim=2)
        weights = self.score(projected + self.source_bias).squeeze(-1).softmax(dim=2)
        weights = weights.masked_fill(~mask.unsqueeze(-1), 0)
        return (weights.unsqueeze(-1) * projected).sum(2), weights


class FusionBiLSTMCRF(nn.Module):
    def __init__(self, source_dims, num_tags, fusion_dim=200, hidden_dim=100, dropout=0.3):
        super().__init__()
        self.fusion = SourceAttention(source_dims, fusion_dim)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(fusion_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(2 * hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, batch, return_attention=False):
        fused, weights = self.fusion(batch['features'], batch['mask'])
        packed = pack_padded_sequence(self.dropout(fused), batch['lengths'].cpu(), batch_first=True, enforce_sorted=False)
        output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(output, batch_first=True, total_length=batch['mask'].size(1))
        emissions = self.classifier(self.dropout(output))
        return (emissions, weights) if return_attention else emissions

    def loss(self, batch):
        return -self.crf(self(batch), batch['tags'], mask=batch['mask'], reduction='mean')

    def decode(self, batch):
        return self.crf.decode(self(batch), mask=batch['mask'])


def train_epoch(model, loader, optimizer, device):
    model.train()
    total, count = 0.0, 0
    for raw in loader:
        batch = move_batch(raw, device)
        optimizer.zero_grad(set_to_none=True)
        loss = model.loss(batch)
        if not torch.isfinite(loss):
            raise RuntimeError('Nonfinite loss')
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item() * len(batch['lengths'])
        count += len(batch['lengths'])
    return total / count


@torch.no_grad()
def evaluate_model(model, loader, labels, device):
    from seqeval.metrics import precision_score, recall_score, f1_score
    from seqeval.scheme import IOB2
    model.eval()
    gold, pred = [], []
    for raw in loader:
        batch = move_batch(raw, device)
        for row, path in enumerate(model.decode(batch)):
            gold.append([labels[i] for i in batch['tags'][row, :len(path)].tolist()])
            pred.append([labels[i] for i in path])
    kwargs = dict(mode='strict', scheme=IOB2, zero_division=0)
    return dict(precision=float(precision_score(gold, pred, **kwargs)),
                recall=float(recall_score(gold, pred, **kwargs)), f1=float(f1_score(gold, pred, **kwargs)))


@torch.no_grad()
def predict_text(text, model, sources, labels, device):
    words, offsets = tokenize_words(text)
    if not words:
        return dict(tokens=[], offsets=[], tags=[], entities=[], attention=[], attention_sources=list(sources))
    features = {name: source.encode(words) for name, source in sources.items()}
    batch = move_batch(collate_words([dict(features=features, tags=torch.zeros(len(words), dtype=torch.long))]), device)
    model.eval()
    emissions, weights = model(batch, return_attention=True)
    tags = [labels[i] for i in model.crf.decode(emissions, mask=batch['mask'])[0]]
    entities, current = [], None
    for (start, end), tag in zip(offsets, tags):
        prefix, _, kind = tag.partition('-')
        if prefix == 'I' and current is not None and current['type'] == kind:
            current['end'] = end
        else:
            if current is not None:
                entities.append(current)
            current = dict(start=start, end=end, type=kind) if prefix in ('B', 'I') else None
    if current is not None:
        entities.append(current)
    for ent in entities:
        ent['text'] = text[ent['start']:ent['end']]
    return dict(tokens=words, offsets=offsets, tags=tags, entities=entities,
                attention=weights[0].cpu().tolist(), attention_sources=model.fusion.names)


In [22]:
# Chỉ khởi tạo và tải các nguồn được chọn; không tải model của nguồn đã tắt.
from huggingface_hub import hf_hub_download
sources = {}
for name in ENABLED_SOURCES:
    if name == 'biomedbert':
        sources[name] = TransformerSource(MODEL_NAME, DEVICE, TRANSFORMER_MAX_LENGTH)
        sources[name].tokenizer.save_pretrained(OUTPUT_DIR / 'tokenizer')
    elif name == 'flair':
        sources[name] = FlairSource(FLAIR_NAMES, DEVICE)
    elif name == 'fasttext':
        if not FASTTEXT_PATH.is_file():
            if not DOWNLOAD_FASTTEXT:
                raise FileNotFoundError(FASTTEXT_PATH)
            FASTTEXT_PATH = Path(hf_hub_download('facebook/fasttext-en-vectors', 'model.bin',
                                               local_dir=str(FASTTEXT_PATH.parent)))
        sources[name] = FastTextSource(FASTTEXT_PATH)
SOURCE_DIMS = {name: source.dim for name, source in sources.items()}
FEATURE_CONFIG = dict(annotation_policy=ANNOTATION_POLICY, sources=ENABLED_SOURCES, transformer=MODEL_NAME, flair=FLAIR_NAMES,
                      fasttext=str(FASTTEXT_PATH), token_pattern=TOKEN_PATTERN,
                      transformer_max_length=TRANSFORMER_MAX_LENGTH, pooling='mean_last_hidden',
                      fasttext_lowercase=True, alignment='regex_exact_boundaries_v1')
MODEL_CONFIG = dict(source_dims=SOURCE_DIMS, num_tags=len(LABELS), fusion_dim=200, hidden_dim=100, dropout=0.3)
# Cache riêng mỗi phiên để không nhầm checkpoint hoặc cấu hình nguồn.
CACHE_DIR = Path(tempfile.mkdtemp(prefix='features_', dir=OUTPUT_DIR))
print('Dimensions:', SOURCE_DIMS)


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

100%|██████████| 69.4M/69.4M [00:04<00:00, 16.2MB/s]

2026-09-23 14:54:00,194 copying /tmp/tmp8teofw0w to cache at /root/.flair/embeddings/pubmed-backward.pt


2026-09-23 14:54:00,234 removing temp file /tmp/tmp8teofw0w


model.bin:   0%|          | 0.00/7.24G [00:00<?, ?B/s]

Dimensions: {'biomedbert': 768, 'flair': 4096, 'fasttext': 300}


In [23]:
def make_loader(rows, shuffle=False):
    dataset = CachedDataset(rows, sources, CACHE_DIR)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
                      collate_fn=collate_words, generator=torch.Generator().manual_seed(SEED))
train_loader = make_loader(prepared['train'], shuffle=True)
dev_loader = make_loader(prepared['dev'])
model = FusionBiLSTMCRF(**MODEL_CONFIG).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
batch = move_batch(next(iter(train_loader)), DEVICE)
with torch.no_grad():
    emissions, attention = model(batch, return_attention=True)
print('Emissions:', emissions.shape, '| Attention [B,T,S]:', attention.shape)
print('Sources:', model.fusion.names)
assert torch.allclose(attention.sum(-1), batch['mask'].float(), atol=1e-6)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


Cache embeddings theo token:   0%|          | 0/6052 [00:00<?, ?it/s]

Cache embeddings theo token:   0%|          | 0/853 [00:00<?, ?it/s]

Emissions: torch.Size([32, 58, 9]) | Attention [B,T,S]: torch.Size([32, 58, 3])
Sources: ['biomedbert', 'flair', 'fasttext']
Trainable parameters: 1319108


In [24]:
best_f1, stale_epochs, history = -1.0, 0, []
BEST_PATH = OUTPUT_DIR / 'best.pt'
for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, train_loader, optimizer, DEVICE)
    dev = evaluate_model(model, dev_loader, LABELS, DEVICE)
    row = dict(epoch=epoch, train_loss=loss, **{f'dev_{k}': v for k, v in dev.items()})
    history.append(row)
    print(row)
    (OUTPUT_DIR / 'history.json').write_text(json.dumps(history, indent=2))
    if dev['f1'] > best_f1:
        best_f1, stale_epochs = dev['f1'], 0
        torch.save(dict(model_state_dict=model.state_dict(), model_config=MODEL_CONFIG,
                        feature_config=FEATURE_CONFIG, labels=LABELS, seed=SEED,
                        epoch=epoch, dev_f1=best_f1, split_manifest=manifest), BEST_PATH)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            break
print('Best dev F1:', best_f1)


{'epoch': 1, 'train_loss': 3.7461544463591943, 'dev_precision': 0.8981042654028436, 'dev_recall': 0.8618533257532689, 'dev_f1': 0.8796054540179866}
{'epoch': 2, 'train_loss': 1.4191086944648152, 'dev_precision': 0.8965317919075144, 'dev_recall': 0.8817509948834565, 'dev_f1': 0.8890799656061908}
{'epoch': 3, 'train_loss': 1.0422063615254353, 'dev_precision': 0.9266826923076923, 'dev_recall': 0.8766344513928368, 'dev_f1': 0.9009640666082384}
{'epoch': 4, 'train_loss': 0.8322977983470949, 'dev_precision': 0.8892655367231639, 'dev_recall': 0.8948266060261513, 'dev_f1': 0.8920374043638424}
{'epoch': 5, 'train_loss': 0.6768151885692599, 'dev_precision': 0.9066820276497696, 'dev_recall': 0.8948266060261513, 'dev_f1': 0.9007153075822604}
{'epoch': 6, 'train_loss': 0.598538924429642, 'dev_precision': 0.906392694063927, 'dev_recall': 0.9027856736782263, 'dev_f1': 0.9045855881515238}
{'epoch': 7, 'train_loss': 0.4920728353183946, 'dev_precision': 0.9143367043426531, 'dev_recall': 0.87379192723138

In [25]:
checkpoint = torch.load(OUTPUT_DIR / 'best.pt', map_location='cpu', weights_only=True)
if checkpoint['feature_config'] != FEATURE_CONFIG or checkpoint['labels'] != LABELS:
    raise ValueError('Checkpoint không khớp nguồn embedding hoặc nhãn')
model = FusionBiLSTMCRF(**checkpoint['model_config']).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
results = dict(dev=evaluate_model(model, dev_loader, LABELS, DEVICE))
# Test chỉ trích embedding/đánh giá sau khi chọn checkpoint bằng dev.
if prepared['test']:
    test_loader = make_loader(prepared['test'])
    results['test'] = evaluate_model(model, test_loader, LABELS, DEVICE)
results['annotation_policy'] = ANNOTATION_POLICY
results['evaluation_target'] = 'projected_token_bio' if ANNOTATION_POLICY == 'project' else 'strict_token_bio'
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(results, indent=2))
print(results)


Cache embeddings theo token:   0%|          | 0/665 [00:00<?, ?it/s]

{'dev': {'precision': 0.9282407407407407, 'recall': 0.9118817509948834, 'f1': 0.9199885288213363}, 'test': {'precision': 0.7450704225352113, 'recall': 0.7711370262390671, 'f1': 0.7578796561604585}, 'annotation_policy': 'project', 'evaluation_target': 'projected_token_bio'}
